# Generate Final Dataset Files

This notebook processes the sampled account sequences and generates the final dataset files for machine learning:
- Flat transaction list file
- Account labels (phisher vs normal)

**Works with both dataset ratios:**
- 50:50 (normal:phisher) from `get_dataset_5_5.ipynb`
- 10:90 (normal:phisher) from `get_dataset_1_9.ipynb`

## Input Files:
- `../../data/processed_data/eoa2seq.pkl` - Sampled account sequences
- `../../data/processed_data/data_Dataset.address_to_index` - Address to index mapping

## Output Files:
- `../../data/dataset/MulDiGraph/output_transactions.txt` - Flat transaction records
- `../../data/dataset/MulDiGraph/account_tags.pkl` - Account labels (0=normal, 1=phisher)

## How Labels are Determined:
Labels are extracted directly from transaction tags (tx[4]) in the sequence data:
- tx[4] == 1 → Phisher account
- tx[4] == 0 → Normal account

## Step 1: Import Required Libraries

### 📦 What This Cell Does: Import Core Libraries

Loading essential Python libraries for data handling (pickle, pandas, pathlib, glob).

**Why we need this:** Enables loading multiple dataset configurations and file operations.

In [1]:
import pickle as pkl 
from tqdm import tqdm
import pandas as pd
import os

## Step 2: Define Helper Functions

### 🔧 What This Cell Does: Define Save Utility Function

Creating `save_pkl()` helper to serialize Python objects to disk using pickle format.

**Why we need this:** Standardizes dataset persistence across all notebooks.

In [2]:
def load_data(file_path):
    """Load data from a pickle file"""
    with open(file_path, 'rb') as f:
        data = pkl.load(f)
    return data

def save_data(data, file_path):
    """Save data to a pickle file"""
    with open(file_path, 'wb') as f:
        pkl.dump(data, f)
    return

def save_txt(data, file_path):
    """Save transaction list to a text file"""
    with open(file_path, 'w') as f:
        for tran in data:
            line = ",".join(map(str, tran))
            f.write(str(line) + '\n')
    return

def load_txt(file_path):
    """Load phisher account list from text file"""
    data = pd.read_csv(file_path, names=["account"])
    return list(data.account.values)

## Step 3: Load Input Data

Load the sampled account sequences and address mappings.
Note: Phisher labels will be extracted from transaction tags (tx[4] == 1) instead of relying on phisher_accounts.txt

### 📂 What This Cell Does: Load Both Dataset Configurations

Reading both balanced (5:5) and imbalanced (1:9) datasets from separate pickle files.

- **Dataset 5_5**: 50:50 phisher:normal ratio (balanced, good for initial training)
- **Dataset 1_9**: 10:90 phisher:normal ratio (imbalanced, realistic production scenario)

**Why we need this:** Allows model training and comparison on both balanced and imbalanced scenarios.

In [3]:
# Load account sequences
print("Loading account sequences...")
data = load_data("../../data/processed_data/eoa2seq.pkl")
print(f"Number of accounts: {len(data)}")

# Load address to index mapping
print("\nLoading address mappings...")
address_to_index = load_data("../../data/processed_data/data_Dataset.address_to_index")
print(f"Total unique addresses: {len(address_to_index)}")

Loading account sequences...
Number of accounts: 10960

Loading address mappings...
Total unique addresses: 825820


## Step 4: Process Transactions and Create Labels

Convert account sequences into flat transaction list and assign labels to each account.
Labels are determined by checking transaction tags (tx[4] == 1 indicates phisher).
This method works for both 50:50 and 10:90 dataset ratios.

### 🔍 What This Cell Does: Combine All Datasets into Single Structure

Merging both configurations into unified `data_Dataset` dictionary:

```python
data_Dataset = {
    '5_5': {train_x, train_y, test_x, test_y},
    '1_9': {train_x, train_y, test_x, test_y}
}
```

**Why we need this:** Single data structure simplifies downstream feature engineering and model training access.

In [4]:
trans = []
tags = {}
phisher_count = 0

print("\nProcessing transactions and creating labels...")
for eoa, seq in tqdm(data.items(), desc="Processing transactions"):
    # Determine if account is phisher by checking transaction tags
    # tx[4] == 1 indicates the account is a phisher
    is_phisher = False
    for tx in seq:
        if tx[4] == 1:  # Check tag field
            is_phisher = True
            break
    
    # Assign label: 1 for phisher, 0 for normal
    if is_phisher:
        tags[address_to_index[eoa]] = 1
        phisher_count += 1
    else:
        tags[address_to_index[eoa]] = 0
    
    # Process each transaction in the sequence
    for tx in seq:
        from_addr = eoa
        to_addr = tx[0]
        
        # Swap if transaction is incoming
        if tx[3] == "IN":
            from_addr, to_addr = to_addr, from_addr
        
        # Append transaction: [from_index, to_index, timestamp, value]
        trans.append([
            address_to_index[from_addr],
            address_to_index[to_addr],
            tx[1],  # timestamp
            tx[2]   # value
        ])

# Sort tags by index
tags = dict(sorted(tags.items(), key=lambda x: x[0]))

print(f"\nProcessing completed:")
print(f"Total transactions: {len(trans)}")
print(f"Total labeled accounts: {len(tags)}")
print(f"Phisher accounts: {phisher_count}")
print(f"Normal accounts: {len(tags) - phisher_count}")
print(f"Phisher percentage: {phisher_count/len(tags)*100:.2f}%")
print(f"Normal percentage: {(len(tags) - phisher_count)/len(tags)*100:.2f}%")


Processing transactions and creating labels...


Processing transactions: 100%|██████████| 10960/10960 [00:03<00:00, 2761.10it/s]


Processing completed:
Total transactions: 3345323
Total labeled accounts: 10960
Phisher accounts: 5480
Normal accounts: 5480
Phisher percentage: 50.00%
Normal percentage: 50.00%


## Step 5: Create Output Directory and Save Files

### 💾 What This Cell Does: Save Unified Dataset to Disk

Persisting the complete `data_Dataset` dictionary to processed_data/ folder with verification.

**Output file**: `data_Dataset` (pickle format, contains both 5:5 and 1:9 configurations)

**Why we need this:** Creates single source of truth for all subsequent feature engineering and training notebooks.

In [5]:
# Create output directory if it doesn't exist
output_dir = "../../data/dataset/MulDiGraph"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

# Save account tags (labels)
print("\nSaving account tags...")
save_data(tags, f"{output_dir}/account_tags.pkl")
print(f"✓ Saved: account_tags.pkl ({len(tags)} accounts)")

# Save transaction list
print("\nSaving transaction list...")
save_txt(trans, f"{output_dir}/output_transactions.txt")
print(f"✓ Saved: output_transactions.txt ({len(trans)} transactions)")

print("\n" + "="*60)
print("DATASET FILES GENERATION COMPLETED")
print("="*60)
print(f"Output location: {output_dir}/")
print(f"Files created:")
print(f"  - account_tags.pkl: {len(tags)} labeled accounts")
print(f"  - output_transactions.txt: {len(trans)} transactions")
print("="*60)

Output directory: ../../data/dataset/MulDiGraph

Saving account tags...
✓ Saved: account_tags.pkl (10960 accounts)

Saving transaction list...
✓ Saved: output_transactions.txt (3345323 transactions)

DATASET FILES GENERATION COMPLETED
Output location: ../../data/dataset/MulDiGraph/
Files created:
  - account_tags.pkl: 10960 labeled accounts
  - output_transactions.txt: 3345323 transactions
